# E9 GNN Navigation

Author: Arush Arora

## Introduction

This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the shortest-distance paths of the graph before expecting it to serve the LLM with **multiplicative** GREPs for navigation tasks, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

### The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

#### Graph Convolutional Network (GNN)

The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Random Graph Positional Encodings (R-PEARL)

The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

### Sparse Graph Transformer

The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q \sim \mathcal{N}(0,\, \mathbf{I})}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

### Transformer

The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

### Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Import modules.
import gc
import copy
import torch
import random
import wandb
import sympy as sp
import networkx as nx

from torch import nn
from torch_geometric.data import Data
from torch.distributions import Cauchy
from torch.nn.utils import clip_grad_norm_
from torch_geometric.utils import to_networkx
from torch.optim.lr_scheduler import ReduceLROnPlateau

from prism.models.r_pearl import RandomGNNPositionalEncodings
from prism.models.gt import GraphTransformer
from prism.models.gcn import GCN
from prism.data import data

In [3]:
# Weights & Biases setup. Mirrors prism.training.train_v3._setup_wandb (project /
# name / tags / group + full-config logging), adapted for this notebook's hand-written
# train loops. Each training stage gets its own run, grouped/tagged by GNN type so the
# R-PEARL and GT variants of the same stage line up on one W&B dashboard. The helpers
# introspect the live optimizer / scheduler / loss objects so EVERY hyperparameter is
# logged without hand-maintaining a list.
WANDB_PROJECT = 'e9-gnn-navigation'


def optimizer_hparams(optimizer):
    """Every optimizer setting: class name, shared defaults, and per-param-group values
    (LRs, betas, eps, weight_decay, ...) with the parameter tensors stripped out."""
    return {
        'optimizer': type(optimizer).__name__,
        'optimizer_defaults': dict(optimizer.defaults),
        'param_groups': [
            {k: v for k, v in g.items() if k != 'params'}
            for g in optimizer.param_groups
        ],
    }


def scheduler_hparams(scheduler):
    """Every LR-scheduler setting (or {'scheduler': None} when unused)."""
    if scheduler is None:
        return {'scheduler': None}
    keys = ('mode', 'factor', 'patience', 'threshold', 'threshold_mode',
            'cooldown', 'min_lrs', 'eps')
    return {
        'scheduler': type(scheduler).__name__,
        **{k: getattr(scheduler, k) for k in keys if hasattr(scheduler, k)},
    }


def loss_hparams(loss_fn):
    """Loss class, reduction, and pos_weight (resolved to plain Python)."""
    out = {'loss_fn': type(loss_fn).__name__,
           'reduction': getattr(loss_fn, 'reduction', None)}
    pos_weight = getattr(loss_fn, 'pos_weight', None)
    if pos_weight is not None:
        out['pos_weight'] = (pos_weight.detach().cpu().tolist()
                             if torch.is_tensor(pos_weight) else pos_weight)
    return out


def init_wandb(stage, hparams):
    """Start a W&B run for a training `stage` ('edge_detection' / 'path_navigation').

    Logs the FULL run config: the GNN construction kwargs (`model_hparams`, set in the
    GNN-instantiation cell) plus every optimizer / scheduler / loss / batching
    hyperparameter the caller assembles in `hparams`. `model_type` selects R-PEARL vs
    GT and drives the run name / tag / group. Returns the run; `reinit=True` so
    successive stages in one notebook session each open a fresh run.
    """
    return wandb.init(
        project=WANDB_PROJECT,
        name=f'{stage}_{model_type}',
        tags=[stage, model_type],
        group=model_type,
        config={'model_type': model_type, 'stage': stage,
                'model': model_hparams, **hparams},
        reinit='return_previous',
    )

In [4]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 3, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [5]:
# Standard options.
eval_path = '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
device = 'cuda'

In [6]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [7]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_009.html


## Experiments

### §1 Pretraining a GNN to Classify Edge Existence

We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify whether an edge exists in the graph or not. Such a model will serve as a backbone pretrained model for fine-tuning on reporting shortest paths. The equations to represent this procedure are below:
$$\mathbf{H} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\mathbf{h}_i\ \Vert\ \mathbf{h}_j\ \Vert\ \mathbf{h}_i \odot \mathbf{h}_j\ \Vert\ |\mathbf{h}_i - \mathbf{h}_j\|\big] \in [0, 1]$$

#### Model Definitions

We first define the models.

In [8]:
# Instantiate a GNN. `model_type` / `model_hparams` are exposed at module scope so
# init_wandb can log the GNN config; create_gnn writes model_hparams as it builds.
def create_gnn(model_type: str):
    global model_hparams
    if model_type == 'gt':
        model_hparams = dict(
            num_layers=3,
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            heads=8,
            num_samples=320,
            dropout=0.1,
            k_pe=3,
            k_gt=2,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = GraphTransformer(**model_hparams)
        gnn.out_features = gnn.d_model
    else:
        model_hparams = dict(
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            num_samples=320,
            dropout=0.1,
            k=3,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = RandomGNNPositionalEncodings(**model_hparams)
        gnn.out_features = gnn.output_projection.out_features
    return gnn


model_type = 'gt'
gnn = create_gnn(model_type)

In [9]:
from typing import Union


# Define a class for edge detection and instantiate it.
class GNNEdgeDetector(nn.Module):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNEdgeDetector, self).__init__()
        self.gnn = gnn
        shape = self.gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(4 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, shape))

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        hi, hj = self.cached_pe[node1], self.cached_pe[node2]
        return self.classifier(torch.cat((hi, hj, hi * hj, abs(hi - hj)), dim=0))
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
detector = GNNEdgeDetector(gnn)

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the graph adjacency given a scene graph PyTorch `Data` object.

In [10]:
# Prepare a graph from the data to be used in the GNN.
from torch_geometric.utils import to_dense_adj, to_networkx
from prism.data import utils
import numpy as np
import sympy as sp

# Prepare the graph for rendition.
ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj = to_dense_adj(ex_graph.edge_index).squeeze().cuda()
ex_graph.edge_index = ex_graph.edge_index.to(device)
ex_graph.x = ex_graph.x.to(device)

EPS = 1e-12
N = ex_graph.num_nodes
ex_graph.paths = torch.zeros((N, N, N)).to(device)
ex_graph.dist = torch.full((N, N), float('inf')).to(device)
g = to_networkx(ex_graph, to_undirected=True, edge_attrs=['distance_m'])
lengths = dict(nx.all_pairs_dijkstra_path_length(g, weight='distance_m'))
for u in range(N):
    for v in lengths[u]:
        ex_graph.dist[u, v] = lengths[u][v]
        for p in nx.all_shortest_paths(g, u, v, weight='distance_m'):
            ex_graph.paths[u, v, p] = 1
            ex_graph.paths[v, u, p] = 1
ex_graph.dist.fill_diagonal_(EPS)

# Show the adjacency matrix of the graph.
render_matrix(adj)

Matrix([
[  0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0],
[1.0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0, 1.0, 1.0,   0, 1.0,   0,   0

In [11]:
# Feed the matrix to the GNN.
gnn.eval()
with torch.no_grad():
    out = gnn(ex_graph).to(device)

_, _, V = torch.pca_lowrank(out, q=10, center=True)
out = out - out.mean(dim=0)
render_matrix(out @ V)

Matrix([
[ 0.173,  0.619,  -0.0151,   0.0557,   0.0357,   0.0573,  -0.0895,   -0.038,   0.0291,  -0.0195],
[ 0.523,  0.642,  -0.0936,    0.255,  -0.0961,  -0.0403,  -0.0477,  -0.0318,  -0.0694,  0.00961],
[ 0.618,  0.664,   -0.117,    0.354,   -0.144,  0.00169,   0.0383,  -0.0107,   0.0456,  -0.0235],
[  1.23, -0.502,   -0.501,    0.141, -0.00318,    0.135,   0.0103,   0.0709,  0.00298,    0.034],
[ -1.48, -0.281,  -0.0711,    0.115,   0.0566,  -0.0537,  -0.0542,   0.0338,   0.0172,  -0.0179],
[ -1.53, -0.305,   -0.144,    0.087,   0.0794,  -0.0683,  -0.0585,   0.0411,   0.0085,   0.0176],
[ 0.105,  0.564,   0.0533,  -0.0545,    0.109,  -0.0902,     0.04,   0.0351,  -0.0267,  -0.0113],
[ 0.136,  0.539,  -0.0385,   -0.149, -0.00127,    0.048,  -0.0276,   0.0299,  -0.0164,  -0.0114],
[ 0.299,  0.623,  -0.0163, -0.00854, -0.00875,  -0.0335,     0.11,  -0.0152,   0.0361,   0.0126],
[ 0.365,  0.572,  -0.0325,   -0.012,  -0.0627,  -0.0656,   0.0703,   0.0367,  -0.0373,  0.00431],
[0.0864,  0

In [12]:
# Test out the Detector.
detector.eval()
with torch.no_grad():
    node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
    out = detector(ex_graph, node1, node2).to(device)

render_matrix(torch.sigmoid(out))

Matrix([[0.519]])

#### Pre-Training of GNN on Edge Incidence

Next, we actually preprocess and train the GNN using the steps defined above.

In [13]:
# Import Modules.
from torch_geometric.loader import DataLoader

# Configure the training and test datasets.
train_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)
test_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
)

# Configure the validation dataset.
train_prop = 0.8
train_num = len(train_dataset)
train_keys = random.sample(list(train_dataset.keys()), k=int(train_num * train_prop))
val_dataset = {k: v for k, v in train_dataset.items() if k not in train_keys}
train_dataset = {k: v for k, v in train_dataset.items() if k in train_keys}

# Add edge existence tuples for edge existence.
def generate_data(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    cauchy_dist = Cauchy(loc=0.0, scale=1.0)
    for graph in graphs:
        N = graph.num_nodes
        graph.x = cauchy_dist.sample((N,)).to(device)
        graph.edge_index = graph.edge_index.to(device)

        # Edges.
        combs = torch.tensor(
            [[u, v] for u in range(N) for v in range(u + 1, N)],
            device=device
        ).T
        existence = torch.tensor(
            [combs[:, i].tolist() in graph.edge_index.T.tolist() for i in range(combs.shape[1])],
            device=device
        )
        graph.exclusion = combs[:, ~existence].to(device)
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), 
            dim=0
        ).to(device)

        # Distances and paths.
        graph.paths = torch.zeros((N, N, N)).to(device)
        graph.dist = torch.full((N, N), float('inf')).to(device)
        g = to_networkx(graph, to_undirected=True, edge_attrs=['distance_m'])
        lengths = dict(nx.all_pairs_dijkstra_path_length(g, weight='distance_m'))
        for u in range(N):
            for v in lengths[u]:
                graph.dist[u, v] = lengths[u][v]
                for p in nx.all_shortest_paths(g, u, v, weight='distance_m'):
                    graph.paths[u, v, p] = 1
                    graph.paths[v, u, p] = 1
        graph.dist.fill_diagonal_(EPS)
        
    return graphs


def reshuffle(graphs):
    for graph in graphs:
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), 
            dim=0
        ).to(device)
    
    return graphs


train_graphs = generate_data(train_dataset)
val_graphs = generate_data(val_dataset)
test_graphs = generate_data(test_dataset)

In [14]:
# Train the GNNEdgeDetector to reconstruct the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 150
es_patience = 5
train_edges = False

def test_loop_edges(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss, correct = 0, 0
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            test_loss += loss_fn(preds, graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds.sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, f1


def train_loop_edges(train_dataloader, val_dataloader, test_dataloader, model,
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_val, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('edge_detection', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        reshuffle(val_dataloader.dataset)
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            val_loss, _ = test_loop_edges(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss.
            graph = train_dataloader.dataset[idx]
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            loss = loss_fn(preds, graph.edges_y)

            # Backpropagation.
            (loss / batch_size).backward()
            model.invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)
    
    # Test the finished model.
    test_loop_edges(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


loss_fn = nn.BCEWithLogitsLoss()
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs, batch_size=batch_size)
if train_edges:
    optimizer = torch.optim.AdamW([
        {'params': detector.gnn.parameters(), 'lr': 3e-5},
        {'params': detector.classifier.parameters(), 'lr': 3e-4},
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_edges(train_dataloader, val_dataloader, test_dataloader, detector,
                     loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler

gc.collect()

250

In [15]:
if train_edges:
    torch.save(detector, '../outputs/e9_multistage_training/edge_detector.pt')
    torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
else:
    detector = torch.load('../outputs/e9_multistage_training/special/edge_detector_final.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/special/edge_detector_{model_type}_final.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence

We now test the trained model on the evaluation dataset. First, we we will render the output for clarity.

In [16]:
# Test out the Detector.
detector.eval().to(device)
with torch.no_grad():
    node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
    out = detector(ex_graph, node1, node2)

render_matrix(torch.sigmoid(out))

Matrix([[5.1e-16]])

In [17]:
# Evaluate the GNN on its reconstruction of test graph adjacencies.
test_loop_edges(test_dataloader, detector, loss_fn)
pass

Test Error: 
 Accuracy: 92.6%, F1: 0.930 | P: 0.874 | R: 0.995 | Bal Acc: 92.6% | Avg loss: 0.235880 



### §2 Fine-tuning the GNN to Classify Shortest-Path Node Inclusion

We now wish to optimize the pre-trained GNN (R-PEARL or Graph Transformer) to classify which nodes reside on the shortest path between two given nodes in the graph. Such a model will serve as the backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbf{\Psi} = \Phi\bigg(\Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big);\, S, \mathcal{H}\bigg)$$
$$c_2(\Psi, \Psi) = \sqrt{2\operatorname{diag}(\Psi^2) - 2\Psi^2} \approx [SPD]$$
$$\mathbf{E} = \mathbb{E}\left[\frac{[SPD]_{ij}}{\delta(i, j)}\right]_{i, j \in [N]}$$

#### Model Definitions

We first define the model by attaching a simple GCN head to the GNN positional encoder.

In [18]:
# Define a class for shortest-path distance estimation and instantiate it.
class GNNShortestPathsEstimator(nn.Module):
    """
    Simple class to predict the shortest-path distance graphical lasso estimator (covariance).
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNShortestPathsEstimator, self).__init__()
        self.head = GCN(
            model_hparams['d_model'],
            model_hparams['d_model'],
            model_hparams['num_layers'],
            skip_connection=True,
            dropout=model_hparams['dropout'],
            k=model_hparams['k_pe']
        )
        self.gate = nn.Parameter(torch.tensor(0.1))
        self.gnn = gnn

    def forward(self, graph: Data):
        graph = graph.clone()
        graph.x = self.gnn(graph)
        out = self.head(graph)
        out = torch.cdist(out, out, p=2)
        return self.gate * out

    def psd(self, dist: torch.Tensor):
        eigh_vals, eigh_vecs = torch.linalg.eigh(dist)
        eigh_vals = torch.sigmoid(eigh_vals)
        return eigh_vecs @ torch.diag(eigh_vals.clamp(min=1e-12, max=1.0)) @ eigh_vecs.T
    
    def inv_psd(self, preds: torch.Tensor):
        eigh_vals, eigh_vecs = torch.linalg.eigh(preds)
        eigh_vals = torch.log(
            (eigh_vals.clamp(min=1e-12, max=1.0)) / (1 - eigh_vals).clamp(min=1e-12, max=1.0)
        )
        return eigh_vecs @ torch.diag(eigh_vals) @ eigh_vecs.T

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the shortest-path distances matrix given a scene graph PyTorch `Data` object.

In [19]:
# Test out the SPD GNN.
spd_gnn = GNNShortestPathsEstimator(detector.gnn).eval()
with torch.no_grad():
    out = spd_gnn(ex_graph).to(device)

render_matrix(out)

Matrix([
[ 0.05,  39.8,  52.9,  82.6, 106.0,  124.0, 217.0, 134.0, 118.0,  69.3, 215.0, 215.0,  63.4, 175.0, 159.0, 144.0, 139.0, 118.0, 176.0, 200.0, 119.0, 156.0,  96.5, 163.0, 430.0, 494.0, 448.0, 267.0, 432.0, 261.0],
[ 39.8,  0.05,  14.4,  60.4, 117.0,  138.0, 256.0, 173.0, 157.0, 108.0, 255.0, 255.0, 102.0, 214.0, 175.0, 157.0, 152.0, 127.0, 193.0, 219.0, 128.0, 171.0, 104.0, 183.0, 457.0, 522.0, 475.0, 291.0, 459.0, 286.0],
[ 52.9,  14.4,     0,  56.9, 123.0,  146.0, 270.0, 187.0, 170.0, 122.0, 268.0, 268.0, 116.0, 228.0, 182.0, 164.0, 158.0, 132.0, 200.0, 227.0, 134.0, 177.0, 110.0, 191.0, 466.0, 532.0, 485.0, 300.0, 469.0, 295.0],
[ 82.6,  60.4,  56.9,     0, 122.0,  143.0, 283.0, 202.0, 187.0, 141.0, 281.0, 281.0, 136.0, 242.0, 144.0, 125.0, 119.0,  91.7, 162.0, 190.0,  92.9, 139.0, 110.0, 187.0, 460.0, 525.0, 479.0, 294.0, 463.0, 289.0],
[106.0, 117.0, 123.0, 122.0,   0.1,   23.6, 231.0, 162.0, 150.0, 120.0, 229.0, 230.0, 117.0, 194.0, 153.0, 142.0, 138.0, 125.0, 166.0, 187.

In [20]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    14.2,    23.6,   146.0,   155.0,   150.0,    1.06,    26.8,    8.42,    22.6,    24.5,    5.67,    20.7,    25.8,   128.0,   136.0,   142.0,   117.0,   135.0,   132.0,   126.0,   143.0,   165.0,   154.0,   152.0,   147.0,   141.0,   148.0,   141.0,   125.0],
[   14.2, 1.0e-12,    25.2,   158.0,   167.0,   162.0,    13.2,    24.2,    5.83,    24.1,    36.6,    17.8,    32.8,    26.4,   140.0,   137.0,   154.0,   118.0,   137.0,   133.0,   123.0,   155.0,   177.0,   166.0,   164.0,   159.0,   153.0,   160.0,   153.0,   137.0],
[   23.6,    25.2, 1.0e-12,   135.0,   177.0,   171.0,    22.6,    35.4,    19.3,    1.06,    36.5,    18.0,    33.0,    37.6,   128.0,   114.0,   137.0,    95.4,   114.0,   110.0,   124.0,   132.0,   186.0,   176.0,   173.0,   169.0,   163.0,   169.0,   162.0,   146.0],
[  146.0,   158.0,   135.0, 1.0e-12,   159.0,   142.0,   145.0,   136.0,   152.0,   134.0,   121.0,   140.0,   133.0,   134.0,    18.0,    36.4,     9.6,    40.0,    21.9,    

In [21]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[5.0e+10,    2.79,  2.24, 0.567,   0.683,    0.824, 204.0,   5.0,  14.0,  3.07,  8.78,     38.0,  3.07, 6.76,  1.25,     1.06, 0.978,   1.01,   1.3,    1.52,  0.946,  1.09, 0.586,  1.06, 2.84,  3.36,     3.17,   1.8, 3.07,      2.1],
[   2.79, 5.0e+10, 0.573, 0.382,   0.699,    0.854,  19.4,  7.15,  27.0,  4.49,  6.95,     14.3,  3.12,  8.1,  1.25,     1.15, 0.985,   1.07,  1.41,    1.64,   1.04,   1.1, 0.589,   1.1, 2.79,  3.28,      3.1,  1.82,  3.0,     2.09],
[   2.24,   0.573,     0,  0.42,   0.699,     0.85,  11.9,  5.28,  8.82, 115.0,  7.34,     14.9,  3.52, 6.05,  1.42,     1.43,  1.16,   1.39,  1.76,    2.06,   1.08,  1.34, 0.591,  1.09,  2.7,  3.16,     2.98,  1.77, 2.89,     2.01],
[  0.567,   0.382,  0.42,     0,    0.77,     1.01,  1.95,  1.48,  1.23,  1.05,  2.32,     2.01,  1.02,  1.8,  7.97,     3.43,  12.4,   2.29,  7.41,    5.59,   2.22,  47.5, 0.761,  1.12, 2.97,  3.77,     3.26,  1.93, 3.18,     1.79],
[  0.683,   0.699, 0.699,  0.77, 1.0e+11,     1.07,   1

#### Definition of a Custom Loss Function: Graphical Lasso Estimator 

We seek to reproduce the [Graphical Lasso Estimator](https://en.wikipedia.org/wiki/Graphical_lasso) custom loss function within the PyTorch framework. Since such an error and gradient computation function requires a differentiable interpretation of the $L_1$ regularization penalty, we must define a new subclass of `torch.autograd.Function` to implement this regression objective within the working environment.

The Graphical Lasso Estimator is defined through the following mathematical optimizer:

$$\hat{\Theta} = \argmax_{\Theta \succ 0} L(\Theta) = \argmax_{\Theta \succ 0}\left(\log\det(\Theta) - \operatorname{tr}(S\Theta) - \lambda\sum_{i, j}|\Theta_{ij}|\right)$$

Thus, it has the following derivative evaluation:

$$\nabla_{\Theta} L(\Theta) = \frac{1}{\det(\Theta)} \det(\Theta) \Theta^{-\top} - S^T - \lambda \begin{cases}1 & \text{if } \Theta_{ij} > 0 \\ 0 & \text{if } \Theta_{ij} = 0 \\ -1 & \text{if } \Theta_{ij} < 0\end{cases}$$
$$\nabla_{\Theta} L(\Theta) = \Theta^{-1} - S - \lambda \operatorname{sign}(\Theta)$$

In [22]:
LAMBDA = 1e-7


class GraphicalLassoEstimator(torch.autograd.Function):
    @staticmethod
    def forward(ctx, preds, targets):
        """
        Computes the loss value for the Graphical Lasso Estimator loss function.
        """
        ctx.save_for_backward(preds, targets)
        sign, logdet = torch.linalg.slogdet(preds)
        assert (sign > 0).all()
        return -logdet + torch.trace(targets @ preds) - LAMBDA * preds.abs().sum()

    @staticmethod
    def backward(ctx, grad_output):
        """
        Computes custom gradients with respect to the inputs. Honors the requirement
        for L1 differentiability within the PyTorch framework.
        """
        preds, targets = ctx.saved_tensors
        grad_predictions = grad_targets = None
        if ctx.needs_input_grad[0]:
            grad_predictions = grad_output * - (preds.inverse() - targets - LAMBDA * preds.sign())
        if ctx.needs_input_grad[1]:
            grad_targets = grad_output * preds
        return grad_predictions, grad_targets

#### Fine-Tuning of GNN on Shortest-Path Distances

We preprocess and train the GNN using the steps defined above.

In [ ]:
# Train the GNN to reconstruct the shortest-path distances of the graph.
batch_size = 4
val_freq = 5
epochs = 200
es_patience = 5
train_dists = False

def test_loop_dists(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model['dists'].to(device).eval()
    model['edges'].to(device).eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss = {'dists': 0, 'edges': 0}
    correct, error_norm = 0, 0
    tp = fp = fn = tn = 0
    
    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            preds = {
                'dists': model['dists'](graph),
                'edges': torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
            }

            test_loss['dists'] += loss_fn['dists'](preds['dists'], graph.dist).item()
            error = preds['dists'] / graph.dist
            error.fill_diagonal_(0)
            error_norm += torch.linalg.matrix_norm(error) / error.shape[0]

            test_loss['edges'] += loss_fn['edges'](preds['edges'], graph.edges_y).mean().item()
            true = graph.edges_y.bool()
            pred = preds['edges'] > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds['edges'].sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss['dists'] /= size
    error_norm /= size
    print(f"Test Error #1: \n Avg error: {error_norm:>0.3f} \n Avg loss: {test_loss['dists']:>8f} \n")

    test_loss['edges'] /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error #2: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss['edges']:>8f} \n")

    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss_dists': test_loss['dists'],
            f'{wandb_prefix}/error_norm': error_norm,
            f'{wandb_prefix}/loss_edges': test_loss['edges'],
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, error_norm


def train_loop_dists(train_dataloader, val_dataloader, test_dataloader, model,
                     loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model['dists'].to(device).train()
    model['edges'].to(device).train()
    val_loss: float = 0
    best_val, best_state, bad_runs, best_state = float('inf'), None, 0, {}
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('shortest_path_distances', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn['dists']),
        **loss_hparams(loss_fn['edges']),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            val_loss, _ = test_loop_dists(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss['dists'])
            if (val_loss['dists'] < best_val['edges'] - 1e-3 
                    and val_loss['edges'] < best_val['edges'] - 1e-3):
                best_val, bad_runs = val_loss, 0
                best_state['dists'] = copy.deepcopy(model['dists'].state_dict())
                best_state['edges'] = copy.deepcopy(model['edges'].state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model['dists'].train
            model['edges'].train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss.
            graph = train_dataloader.dataset[idx]
            preds = {
                'dists': model['dists'](graph),
                'edges': torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
            }
            loss = {
                'dists': loss_fn['dists'](preds['dists'], graph.dist),
                'edges': loss_fn['edges'](preds['edges'], graph.edges_y)
            }

            # Backpropagation.
            ((loss['dists'] / graph.num_nodes) + (loss['edges'] / batch_size)).backward()
            model['edges'].invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model['dists'].parameters(), max_norm=1.0)
                clip_grad_norm_(model['edges'].parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                current = j
                wandb.log({
                    'train/loss_dists': loss['dists'],
                    'train/loss_edges': loss['edges'],
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss #1: {loss['dists'].item():>7f}  [{current:>5d}/{size:>5d}]")
                print(f"Loss #2: {loss['edges'].item():>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model['dists'].load_state_dict(best_state['dists'])
        model['edges'].load_state_dict(best_state['edges'])

    # Test the finished model.
    test_loop_dists(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish MSE/Graphical-Lasso loss.
loss_fn = {
    'dists': nn.MSELoss(), # GraphicalLassoEstimator.apply
    'edges': nn.BCEWithLogitsLoss()
}
if train_dists:
    optimizer = torch.optim.AdamW([
        {'params': gnn.parameters(), 'lr': 3e-5},
        {'params': detector.classifier.parameters(), 'lr': 3e-4},
        {'params': spd_gnn.head.parameters(), 'lr': 3e-4},
        {'params': [spd_gnn.gate], 'lr': 3e-4}
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_dists(train_dataloader, val_dataloader, test_dataloader, 
                    {'dists': spd_gnn, 'edges': detector}, loss_fn, optimizer, 
                    scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler

gc.collect()

0

In [24]:
if train_dists:
    torch.save(detector, '../outputs/e9_multistage_training/edge_detector.pt')
    torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
else:
    detector = torch.load('../outputs/e9_multistage_training/edge_detector.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt'))

In [25]:
if train_dists:
    torch.save(spd_gnn, '../outputs/e9_multistage_training/spd_gnn.pt')
    torch.save(spd_gnn.gnn.state_dict(), f'../outputs/e9_multistage_training/spd_gnn_{model_type}.pt')
else:
    spd_gnn = torch.load(f'../outputs/e9_multistage_training/spd_gnn.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/spd_gnn_{model_type}.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence and Shortest-Paths Distance Estimation

We test the pre-trained model on the evaluation dataset. We we will render the output error matrix $\mathbf{E}$ for visibility.

In [26]:
# Test out the SPD GNN.
spd_gnn.eval().to(device)
with torch.no_grad():
    out = spd_gnn(ex_graph)

render_matrix(out)

Matrix([
[    0,   23.2,   23.0,  85.2, 143.0, 135.0,   41.6,  31.4,  33.1,  26.3,  43.0,   40.9,   16.6,  35.3,  92.5,   89.2,  92.0,  84.8,   93.3,   94.0,  85.7,   93.3, 129.0, 152.0, 163.0,  167.0, 156.0, 156.0,  156.0, 156.0],
[ 23.2, 0.0539,   14.2,  94.1, 158.0, 150.0,   47.2,  44.8,  36.8,  29.1,  53.4,   49.3,   36.3,  48.2,  98.2,   96.9,  98.3,  93.4,   98.8,   99.4,  94.1,   98.3, 144.0, 168.0, 180.0,  184.0, 173.0, 172.0,  173.0, 172.0],
[ 23.0,   14.2, 0.0539,  83.9, 152.0, 144.0,   46.4,  42.0,  33.6,  24.8,  52.7,   47.9,   34.4,  47.3,  88.3,   87.1,  88.5,  83.7,   88.8,   89.5,  84.5,   88.1, 137.0, 161.0, 174.0,  179.0, 168.0, 166.0,  167.0, 166.0],
[ 85.2,   94.1,   83.9,     0, 113.0, 105.0,   89.9,  74.0,  82.2,  84.6,  83.1,   84.9,   78.2,  80.4,  23.3,   23.5,  20.0,  16.8,   25.0,   30.4,  17.2,   27.6,  96.8, 124.0, 138.0,  144.0, 132.0, 130.0,  131.0, 131.0],
[143.0,  158.0,  152.0, 113.0, 0.027,  12.6,  159.0, 135.0, 157.0, 157.0, 145.0,  151.0,  134.0, 13

In [27]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    14.2,    23.6,   146.0,   155.0,   150.0,    1.06,    26.8,    8.42,    22.6,    24.5,    5.67,    20.7,    25.8,   128.0,   136.0,   142.0,   117.0,   135.0,   132.0,   126.0,   143.0,   165.0,   154.0,   152.0,   147.0,   141.0,   148.0,   141.0,   125.0],
[   14.2, 1.0e-12,    25.2,   158.0,   167.0,   162.0,    13.2,    24.2,    5.83,    24.1,    36.6,    17.8,    32.8,    26.4,   140.0,   137.0,   154.0,   118.0,   137.0,   133.0,   123.0,   155.0,   177.0,   166.0,   164.0,   159.0,   153.0,   160.0,   153.0,   137.0],
[   23.6,    25.2, 1.0e-12,   135.0,   177.0,   171.0,    22.6,    35.4,    19.3,    1.06,    36.5,    18.0,    33.0,    37.6,   128.0,   114.0,   137.0,    95.4,   114.0,   110.0,   124.0,   132.0,   186.0,   176.0,   173.0,   169.0,   163.0,   169.0,   162.0,   146.0],
[  146.0,   158.0,   135.0, 1.0e-12,   159.0,   142.0,   145.0,   136.0,   152.0,   134.0,   121.0,   140.0,   133.0,   134.0,    18.0,    36.4,     9.6,    40.0,    21.9,    

In [28]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[    0,     1.63,    0.971, 0.585,   0.922, 0.898,     39.1,  1.17,  3.93,  1.17,  1.75,     7.21,    0.802,  1.37, 0.724,    0.657, 0.647,  0.725,    0.691,    0.714,  0.681,    0.653, 0.781,   0.989,  1.07,     1.14,  1.11,  1.06,      1.1,  1.25],
[ 1.63, 5.39e+10,    0.563, 0.596,   0.947, 0.928,     3.58,  1.85,  6.32,  1.21,  1.46,     2.77,     1.11,  1.83, 0.702,    0.705, 0.637,  0.789,    0.724,    0.746,  0.764,    0.635, 0.816,    1.01,   1.1,     1.16,  1.13,  1.08,     1.13,  1.26],
[0.971,    0.563, 5.39e+10,  0.62,   0.857, 0.839,     2.05,  1.19,  1.74,  23.4,  1.44,     2.67,     1.04,  1.26, 0.691,    0.762, 0.647,  0.878,    0.782,    0.812,  0.682,    0.665, 0.737,   0.919,   1.0,     1.06,  1.03, 0.979,     1.03,  1.14],
[0.585,    0.596,     0.62,     0,   0.712, 0.735,    0.621, 0.543,  0.54,  0.63, 0.686,    0.606,    0.588,   0.6,  1.29,    0.647,  2.09,  0.421,     1.14,    0.895,  0.412,     9.44, 0.672,   0.746, 0.892,     1.03, 0.898, 0.853,    0.

In [29]:
def are_models_equal(model1, model2):
    # 1. Check if both models have the exact same state_dict keys
    if model1.state_dict().keys() != model2.state_dict().keys():
        return False
    
    # 2. Check if all parameters and buffers are exactly equal
    for key, value1 in model1.state_dict().items():
        value2 = model2.state_dict()[key]
        
        # Use torch.equal for strict element-wise and structural equality
        if not torch.equal(value1, value2):
            return False
            
    return True

are_models_equal(detector.gnn, spd_gnn.gnn)

True

In [30]:
# Evaluate the GNN on its reconstruction of test graph edge incidences and shortest-path distances together.
models = {'dists': spd_gnn, 'edges': detector}
test_loop_dists(test_dataloader, models, loss_fn)
pass

Test Error #1: 
 Avg error: 2.677 
 Avg loss: 1284.375833 

Test Error #2: 
 Accuracy: 94.7%, F1: 0.948 | P: 0.911 | R: 0.989 | Bal Acc: 94.6% | Avg loss: 0.213575 



### §3 Fine-tuning the GNN to Predict Shortest-Path Subgraph Adjacencies

We now wish to optimize the pre-trained GNN (R-PEARL or Graph Transformer) to classify which nodes reside on the shortest path between two given nodes in the graph. Such a model will serve as the backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H}\,)\big];\, \mathcal{T}\Big)$$
$$\mathbf{\hat{y}}_{ij} = \Phi\Big(\big[\Psi \,\Vert\, \mathbf{h}_i \cdot \mathbf{1}^T \,\Vert\, \mathbf{h}_j \cdot \mathbf{1}^T \,\Vert\, \Psi \cdot \mathbf{h}_i \,\Vert\, \Psi \cdot \mathbf{h}_j\big];\, S, \mathcal{H}\Big) \in [0, 1]^N$$

In [31]:
from typing import Union


# Define a class for edge detection and instantiate it.
class GNNShortestPathNavigator(GNNEdgeDetector):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super().__init__(gnn)
        shape = gnn.out_features
        self.head = GCN(
            5*model_hparams['d_model'],
            model_hparams['d_model'],
            model_hparams['num_layers'],
            skip_connection=True,
            dropout=model_hparams['dropout'],
            k=model_hparams['k_pe']
        )
        self.classifier = nn.Linear(in_features=shape, out_features=1)

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        pe = self.cached_pe
        hi, hj = pe[node1], pe[node2]
        
        graph = graph.clone()
        features = torch.cat(
            (pe, hi.expand_as(pe), hj.expand_as(pe), pe * hi, pe * hj), dim=1
        )
        graph.x = features
        return self.classifier(self.head(graph))
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
navigator = GNNShortestPathNavigator(spd_gnn.gnn)

In [32]:
# Test out the Navigator.
navigator.eval().to(device)
with torch.no_grad():
    node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
    out = navigator(ex_graph, node1, node2)

render_matrix(torch.sigmoid(out))

Matrix([
[ 2.11e-8],
[ 1.76e-5],
[ 6.16e-5],
[ 2.09e-6],
[   0.699],
[   0.624],
[4.13e-21],
[1.87e-16],
[1.26e-15],
[ 5.2e-12],
[ 8.6e-22],
[7.34e-22],
[4.02e-12],
[3.93e-19],
[5.18e-16],
[1.71e-14],
[2.42e-14],
[1.67e-12],
[2.65e-17],
[9.62e-19],
[1.43e-12],
[4.12e-15],
[   0.477],
[    0.92],
[   0.999],
[     1.0],
[   0.734],
[   0.999],
[   0.676],
[   0.992]])

#### Fine-Tuning of GNN on Shortest-Path Node Inclusion
Finally, we preprocess and train the GNN using the steps defined above.

In [ ]:
# Train the GNN to reconstruct the eigenvectors of the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 1
es_patience = 5
detour_bce = False

# Loss function.
class BCEWithL1Loss(nn.Module):
    def __init__(self, pos_weight, l1_lambda):
        super(BCEWithL1Loss, self).__init__()
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device), reduction='mean')
        self.l1_lambda = l1_lambda

    def forward(self, logits, targets, model):
        bce = self.bce_loss(logits, targets)
        l1_penalty = 0.0
        for name, param in model.named_parameters():
            if 'weight' in name:
                l1_penalty += torch.norm(param, p=1)
        
        total_loss = bce + self.l1_lambda * l1_penalty
        return total_loss

def test_loop_paths(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss = tp = fp = fn = tn = 0

    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            for u in range(graph.num_nodes):
                preds = torch.stack(
                    [model(graph, u, v) for v in range(graph.num_nodes)]
                ).squeeze(-1).to(device)
                test_loss += loss_fn(preds, graph.paths[u], model).mean().item() / graph.num_nodes
                true = graph.paths[u].bool()
                pred = preds > 0
                tp += (pred & true).sum().item()
                fp += (pred & ~true).sum().item()
                fn += (~pred & true).sum().item()
                tn += (~pred & ~true).sum().item()

    test_loss /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n F1: {f1:.3f} | P: {precision:.3f} | R: {recall:.3f} | "
          f"Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, f1


def train_loop_paths(train_dataloader, val_dataloader, test_dataloader, model, 
                     loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    f1: float = 0
    best_val, best_state, bad_runs = -float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('path_navigation', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        'detour_bce': detour_bce,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            val_loss, f1 = test_loop_paths(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best F1 {best_val:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        for j, idx in enumerate(order.tolist()):
            # Compute path metrics for Detour-BCE loss.
            graph = train_dataloader.dataset[idx]
            N = graph.num_nodes
            if detour_bce:
                D = graph.dist
                diam = D[torch.isfinite(D)].max()
                delta = (D[:, None, :] + D.transpose(0, 1)[None, :, :] - D[:, :, None]).clamp_min(0)
                delta_norm = (delta / diam).nan_to_num(0.0)
                reachable = torch.isfinite(D)

            # Compute prediction and loss.
            loss = 0
            for u in range(N):
                preds = torch.stack(
                    [model(graph, u, v) for v in range(N)]
                ).squeeze(-1).to(device)

                # Compute Detour-BCE loss.
                raw_loss = loss_fn(preds, graph.paths[u], model)
                if detour_bce:
                    neg_weights = 1.0 + delta_norm[u]
                    weights = torch.where(
                        graph.paths[u].bool(), torch.ones_like(neg_weights), neg_weights
                    )
                    mask = reachable[u].unsqueeze(-1).float()
                loss = loss + raw_loss / graph.num_nodes

            # Backpropagation.
            (loss / batch_size).backward()
            model.invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item() / N, j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test the finished model.
    test_loop_paths(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish path-vector (sparsity-) sensitive BCE Logit loss.
positive = sum(g.paths.sum() for g in train_graphs)
pos_weight = (sum(g.paths.numel() for g in train_graphs) - positive) / positive
pos_weight **= 0.5
loss_fn = BCEWithL1Loss(pos_weight=pos_weight.to(device), l1_lambda=1e-2)
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs, batch_size=batch_size)
optimizer = torch.optim.AdamW([
    {'params': navigator.gnn.parameters(), 'lr': 3e-5},
    {'params': navigator.head.parameters(), 'lr': 3e-4},
    {'params': navigator.classifier.parameters(), 'lr': 3e-4},
], betas=(0.9, 0.95), weight_decay=0.05)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
train_loop_paths(train_dataloader, val_dataloader, test_dataloader, navigator,
                 loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)
del optimizer, scheduler
gc.collect()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/arushar/.netrc.
wandb: Currently logged in as: arushar (alelab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Validation #0
Test Error: 
 F1: 0.190 | P: 0.126 | R: 0.391 | Bal Acc: 50.1% | Avg loss: 6239.716869 

Epoch #0
Loss: 194.578186  [    3/   32]
Loss: 256.288940  [    7/   32]
Loss: 230.621286  [   11/   32]
Loss: 230.198141  [   15/   32]
Loss: 231.500157  [   19/   32]
Loss: 215.192972  [   23/   32]
Loss: 196.042087  [   27/   32]
Loss: 210.619292  [   31/   32]
Test Error: 
 F1: 0.228 | P: 0.129 | R: 1.000 | Bal Acc: 50.0% | Avg loss: 6138.496642 



epoch,▁▁▁▁▁▁▁▁▁
global_step,▁▂▃▄▅▆▇█
test/bal_acc,▁
test/f1,▁
test/loss,▁
test/precision,▁
test/recall,▁
train/loss,▁█▅▅▅▃▁▃
train/lr,▁▁▁▁▁▁▁▁
val/bal_acc,▁
+4,...


52

In [34]:
torch.save(navigator, '../outputs/e9_multistage_training/path_navigator.pt')
torch.save(navigator.gnn.state_dict(), f'../outputs/e9_multistage_training/path_navigator_{model_type}.pt')
# navigator = torch.load('../outputs/e9_multistage_training/path_navigator.pt', weights_only=False)
# gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/path_navigator_{model_type}.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence
We thus test the fine-tuned model on the evaluation dataset. First, we we will render the output for clarity.

In [35]:
# Test out the Navigator.
navigator.eval().to(device)
with torch.no_grad():
    node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
    out = navigator(ex_graph, node1, node2)

render_matrix(torch.sigmoid(out))

Matrix([
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0],
[1.0]])

In [36]:
# Evaluate the GNN on its reconstruction of test graph shortest paths.
test_loop_paths(test_dataloader, navigator, loss_fn)
pass

Test Error: 
 F1: 0.228 | P: 0.129 | R: 1.000 | Bal Acc: 50.0% | Avg loss: 6138.874271 



In [37]:
# Evaluate the GNN on its reconstruction of test graph shortest path distances.
spd_gnn.gnn.load_state_dict(navigator.gnn.state_dict())
test_loop_dists(
    test_dataloader, {'dists': spd_gnn, 'edges': detector},
    {'dists': nn.MSELoss(), 'edges': nn.BCEWithLogitsLoss()}
)
pass

Test Error #1: 
 Avg error: 3.103 
 Avg loss: 2973.290857 

Test Error #2: 
 Accuracy: 94.0%, F1: 0.942 | P: 0.903 | R: 0.986 | Bal Acc: 94.0% | Avg loss: 0.235965 

